In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

# Фиксирую seed и проверяю доступность cuda
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Доступно: {device}")

# Устанавливаю гиперпараметры
BATCH_SIZE = 32
NUM_EPOCHS = 30
LR = 1e-4
TARGET = 'type' # на этапе baseline решения фокусируюсь на type

Доступно: cuda


In [2]:
# Загружаю датасет, если он был сохранен в блоке с EDA
output_df_f = 'output/annotations.pkl'

if os.path.exists(output_df_f):
    df = pd.read_pickle(output_df_f)
    print(f"Датасет с {len(df)} значений загружен.")
else:
    print(f".pkl не найден по пути {output_df_f}.")

df_clean = df[df['floors'] <= 50].copy()

Датасет с 16921 значений загружен.


In [3]:
# Извлекаю уникальные классы и делаю энкодер
classes = sorted(df_clean[TARGET].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

In [4]:
# Выполняю разделение train, val, test
train_df, tmp_df = train_test_split(
    df_clean, test_size=0.3, stratify=df_clean[TARGET], random_state=42
)
val_df, test_df = train_test_split(
    tmp_df, test_size=0.5, stratify=tmp_df[TARGET], random_state=42
)

print(f"train: {len(train_df)}; val: {len(val_df)}; test: {len(test_df)}")

train: 11824; val: 2534; test: 2534


**Для борьбы с дисбалансом классов принимаем, что вес каждого класса обратно пропорционален его частоте в обучающей выборке.**

In [5]:
counts = df_clean[TARGET].value_counts()
num_classes = len(classes)

class_weights = torch.FloatTensor(
    [len(df_clean) / (num_classes * counts[c]) for c in classes]
).to(device)

In [6]:
class BuildingDataset(Dataset):
    """Датасет torch"""
    
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.tf = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, i):
        # Привожу к RGB во избежание ошибок
        img = Image.open(self.df.loc[i, 'filepath']).convert('RGB')
        
        # Labeling
        lbl = class_to_idx[self.df.loc[i, TARGET]]
        
        return self.tf(img) if self.tf else img, lbl

**В части аугментации делаю следуюющее:**  
**Масштабирую до 256 пикселей и случайно вырезаю квадрат 224x224;**  
**Выполняю случайное отражение по горизонтали (p=0.5);**
**Случайные искажения цвета.**  
*Примененнные способы в целом достаточно распространены и были на маголего по CV.*

In [7]:
# Нормализация для ImageNet
norm = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)


train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    norm
])

# Оценочные, обрезка строго по центру
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    norm
])

# Применяю трансформации выше
train_ds = BuildingDataset(train_df, train_tf)
val_ds = BuildingDataset(val_df, eval_tf)
test_ds = BuildingDataset(test_df, eval_tf)

**Дополнительно для борьбы с дисбалансом использую WeightedRandomSampler**  
*Повышает частоту попадания малочисленных классов в обучающие пакеты, отлично комбинируется с пунктом выше.*  

In [8]:
lbls = [class_to_idx[train_df.iloc[i][TARGET]] for i in range(len(train_df))]
sw = [1.0 / Counter(lbls)[l] for l in lbls]
sampler = WeightedRandomSampler(sw, len(sw), replacement=True)

In [9]:
train_loader = DataLoader(train_ds, BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=0)

In [10]:
# Преодобученная модель
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

**Для baseline решения я использую ResNet-50 обученную на ImageNet, но заменяю head на свой слой с dropout.**  

In [11]:
# Отключаю 30% нейронов и в линейном слое беру целевое число классов
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, num_classes)
)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Полученное число параметров: {total_params}")

Полученное число параметров: 23528522


In [12]:
# Сильнее штрафует редкие классы
criterion = nn.CrossEntropyLoss(weight=class_weights)

# AdamW с L2-регуляризацией
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=LR)

# Планировщик - снижает lr в 2 раза, если точность не растет 3 эпохи к ряду
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

In [14]:
best_acc = 0.0
patience = 7
no_improve = 0

for epoch in range(NUM_EPOCHS):
    model.train()
    tr_loss, tr_corr = 0, 0
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        # Обнуление градиентов, проход, потери, обратный ход, обновление весов
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
        tr_loss += loss.detach().item() * x.size(0)
        tr_corr += (out.argmax(1) == y).sum().item()
    
    tr_loss /= len(train_ds)
    tr_acc = tr_corr / len(train_ds)
    
    # Валидация
    model.eval()
    val_loss, val_corr = 0, 0
    
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            val_loss += loss.item() * x.size(0)
            val_corr += (out.argmax(1) == y).sum().item()
    
    val_loss /= len(val_ds)
    val_acc = val_corr / len(val_ds)
    
    # Вызов планировщика, снижать lr или нет
    scheduler.step(val_acc)
    
    print(f"Эпоха {epoch+1:2d}; train loss={tr_loss:.3f}, train acc={tr_acc:.3f}; val loss={val_loss:.3f}, val acc={val_acc:.3f}")
    # Сохраняяю лучший вариант
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_baseline.pth")
        print(f"Сохранена лучшая модель с val acc={val_acc:.3f}")
        no_improve = 0
    else:
        no_improve += 1
        
    if no_improve >= patience:
        print(f"Предел эпох без улучшения, остановка на {epoch+1}.")

print(f"Обучение завершено, лучшая точность на валидации: {best_acc:.3f}.")

Эпоха  1; train loss=0.692, train acc=0.634; val loss=1.220, val acc=0.501
Сохранена лучшая модель с val acc=0.501
Эпоха  2; train loss=0.516, train acc=0.715; val loss=1.194, val acc=0.547
Сохранена лучшая модель с val acc=0.547
Эпоха  3; train loss=0.409, train acc=0.762; val loss=1.172, val acc=0.584
Сохранена лучшая модель с val acc=0.584
Эпоха  4; train loss=0.328, train acc=0.803; val loss=1.175, val acc=0.602
Сохранена лучшая модель с val acc=0.602
Эпоха  5; train loss=0.273, train acc=0.826; val loss=1.146, val acc=0.616
Сохранена лучшая модель с val acc=0.616
Эпоха  6; train loss=0.247, train acc=0.841; val loss=1.278, val acc=0.610
Эпоха  7; train loss=0.227, train acc=0.853; val loss=1.202, val acc=0.625
Сохранена лучшая модель с val acc=0.625
Эпоха  8; train loss=0.203, train acc=0.863; val loss=1.248, val acc=0.646
Сохранена лучшая модель с val acc=0.646
Эпоха  9; train loss=0.175, train acc=0.879; val loss=1.251, val acc=0.675
Сохранена лучшая модель с val acc=0.675
Эпоха

In [15]:
# Проверка на тесте
model.load_state_dict(torch.load("best_baseline.pth", weights_only=True))
model.eval()

preds, trues = [], []
with torch.no_grad():
    for x, y in test_loader:
        # индексы с макс вероятностью
        preds.extend(model(x.to(device)).argmax(1).cpu().numpy())
        trues.extend(y.numpy())

test_acc = np.mean(np.array(preds) == np.array(trues))
print(f"\nТочность на test: {test_acc:.4f} ({test_acc*100:.1f}%)")

print("\nClassification report:")
print(classification_report(trues, preds, target_names=classes, digits=3))


Точность на test: 0.7182 (71.8%)

Classification report:
              precision    recall  f1-score   support

  apartments      0.862     0.727     0.789       784
   education      0.623     0.518     0.566       137
      garage      0.667     0.706     0.686        51
       hotel      0.591     0.624     0.607       109
       house      0.729     0.823     0.773       362
  industrial      0.657     0.698     0.677       189
      office      0.673     0.779     0.722       389
      public      0.568     0.470     0.514       134
   religious      0.763     0.837     0.798       104
      retail      0.630     0.698     0.662       275

    accuracy                          0.718      2534
   macro avg      0.676     0.688     0.679      2534
weighted avg      0.725     0.718     0.718      2534

